## 1. Environment Setup

In [ ]:
# # 1) Clean old installs
!pip uninstall -y bitsandbytes torch torchvision torchaudio

# 2) Install PyTorch built for CUDA 12.1
!pip install --index-url https://download.pytorch.org/whl/cu121 \
  torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1

# 3) Install a matching bitsandbytes
!pip install bitsandbytes==0.43.1

# 4) (Optional but good) Align the rest
!pip install -U transformers==4.44.2 accelerate==0.34.2 peft==0.11.1


In [ ]:
# # Install required packages
# !pip install -q transformers==4.36.0
!pip install -q datasets==2.14.0
# !pip install -q peft==0.7.1
# !pip install -q accelerate==0.25.0
# !pip install -q wandb==0.16.0
!pip install -q trl==0.7.4
# !pip install -q torch>=2.0.0
# !pip install -q matplotlib seaborn pandas scipy
# ! pip install -U "transformers==4.41.2" "accelerate==0.30.1" \
#   "peft==0.9.0" "bitsandbytes==0.41.1"

In [ ]:
! pip install -U bitsandbytes

In [ ]:
import os
import json
import torch
import wandb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from pathlib import Path
import gc
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)
from datasets import load_dataset, Dataset

# Set random seeds for reproducibility
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clear any existing models/tensors
torch.cuda.empty_cache()
gc.collect()

# Check available memory
if torch.cuda.is_available():
    print(f"GPU Memory before: {torch.cuda.memory_allocated(0)/1e9:.2f} GB allocated")
    print(f"GPU Memory reserved: {torch.cuda.memory_reserved(0)/1e9:.2f} GB reserved")

    # Force clear everything
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

    print(f"GPU Memory after clear: {torch.cuda.memory_allocated(0)/1e9:.2f} GB allocated")

In [ ]:
from huggingface_hub import login, create_repo
hf_token = ''
login(token = hf_token)

## 2. Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
MODEL_CONFIGS = {
    "pythia-2.8b": {
        "model_name": "EleutherAI/pythia-2.8b",
        "lora_r": 16,
        "lora_alpha": 32,
        "batch_size": 8,
        "gradient_accumulation_steps": 2
    },
    "phi-1.5": {
        "model_name": "microsoft/phi-1_5",
        "lora_r": 8,
        "lora_alpha": 16,
        "batch_size": 4,
        "gradient_accumulation_steps": 4
    },
    "gpt-neo-1.3b": {
        "model_name": "EleutherAI/gpt-neo-1.3B",
        "lora_r": 8,
        "lora_alpha": 16,
        "batch_size": 4,
        "gradient_accumulation_steps": 2
    },
    "flan-t5-base": {
        "model_name": "google/flan-t5-base",
        "lora_r": 8,
        "lora_alpha": 16,
        "batch_size": 8,
        "gradient_accumulation_steps": 2,
        "is_encoder_decoder": True
    }
}

SELECTED_MODEL = "pythia-2.8b"
config = MODEL_CONFIGS[SELECTED_MODEL]

# Data paths 
TRAIN_DATA_PATH = "/content/drive/MyDrive/thesis_data/step_10/train_augmented_replay.jsonl"
VAL_DATA_PATH = "/content/drive/MyDrive/thesis_data/step_10/val.jsonl"

# Current step
CURRENT_STEP = 10
TOTAL_STEPS = 10

# Training hyperparameters
LEARNING_RATE = 3e-5
NUM_EPOCHS = 1
MAX_LENGTH = 512
WARMUP_RATIO = 0.1
LORA_DROPOUT = 0.1

# save repo id
repo_id_pythia = "ehoangsimon/pythia-2_8b-replay-step10"

# Output directories
OUTPUT_DIR = f"./outputs/{SELECTED_MODEL}/step_{CURRENT_STEP}"
CHECKPOINT_DIR = f"./checkpoints/{SELECTED_MODEL}/step_{CURRENT_STEP}"

# WandB configuration
WANDB_PROJECT = "supreme-court-continual-learning"
WANDB_RUN_NAME = f"{SELECTED_MODEL}_step_{CURRENT_STEP}_replay"

# Create directories
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Selected Model: {config['model_name']}")
print(f"Current Step: {CURRENT_STEP}/{TOTAL_STEPS}")
print(f"Schema: Simple Question-Answer (no instructions)")
print(f"Output Directory: {OUTPUT_DIR}")

## 3. Initialize WandB

In [ ]:
from huggingface_hub import login

hf_token = ''
login(token=hf_token)

# Login to WandB
wandb.login(key="")

# Initialize WandB run
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "model_name": config["model_name"],
        "schema": "question_answer",
        "step": CURRENT_STEP,
        "total_steps": TOTAL_STEPS,
        "learning_rate": LEARNING_RATE,
        "num_epochs": NUM_EPOCHS,
        "batch_size": config["batch_size"],
        "gradient_accumulation_steps": config["gradient_accumulation_steps"],
        "lora_r": config["lora_r"],
        "lora_alpha": config["lora_alpha"],
        "lora_dropout": LORA_DROPOUT,
        "max_length": MAX_LENGTH,
    },
    tags=[SELECTED_MODEL, f"step_{CURRENT_STEP}", "lora", "continual_learning", "qa_format"]
)

## 4. Load and Prepare Data

In [ ]:
def load_jsonl(file_path):
    """Load JSONL file."""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

def validate_qa_schema(data_sample):
    """Validate and standardize to Q&A format."""

    # Already in Q&A format
    if "question" in data_sample and "answer" in data_sample:
        return "qa", data_sample

    # Convert from instruction format
    if "instruction" in data_sample and "input" in data_sample and "output" in data_sample:
        return "instruction_converted", {
            "question": data_sample["input"],
            "answer": data_sample["output"]
        }

    # Convert from prompt-completion
    if "prompt" in data_sample and "completion" in data_sample:
        return "prompt_completion_converted", {
            "question": data_sample["prompt"],
            "answer": data_sample["completion"]
        }

    raise ValueError(f"Unknown schema. Expected 'question'/'answer' fields. Got: {data_sample.keys()}")

# Load data
train_data_raw = load_jsonl(TRAIN_DATA_PATH)
print(f"Loaded {len(train_data_raw)} training examples")
val_data_raw = load_jsonl(VAL_DATA_PATH)
print(f"Loaded {len(val_data_raw)} validation examples")


# Convert all data to Q&A format
train_data = [validate_qa_schema(item)[1] for item in train_data_raw]
val_data = [validate_qa_schema(item)[1] for item in val_data_raw]


# Convert to HuggingFace datasets
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"\nDataset sizes:")
print(f"Training: {len(train_dataset)}")
print(f"Validation: {len(val_dataset)}")

# Log data statistics to WandB
wandb.log({
    "data/train_size": len(train_dataset),
    "data/val_size": len(val_dataset),
})

## 5. Load Model and Tokenizer

In [ ]:
# Load from a local snapshot of the HF repo to avoid template lookups
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model_id = "ehoangsimon/pythia-2_8b-replay-step9"

# for qunt
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)


# Load tokenizer
print(f"Loading tokenizer from snapshot for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True,trust_remote_code=True)

# Set padding token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("Set tokenizer.pad_token to tokenizer.eos_token")


# Ensure model has a pad_token_id if tokenizer needed one
if getattr(model.config, "pad_token_id", None) is None and tokenizer.pad_token_id is not None:
    model.config.pad_token_id = tokenizer.pad_token_id
    print("Set model.config.pad_token_id to tokenizer.pad_token_id")

# Prepare model for k-bit training (only needed if you are going to LoRA/QLoRA etc.)
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig, TaskType
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False  # you already do this
model.gradient_checkpointing_enable()
# This line is crucial with gradient checkpointing + LoRA
model.enable_input_require_grads()

## 6. Configure LoRA

In [ ]:
# Determine target modules
def get_target_modules(model_name):
    model_name_lower = model_name.lower()
    if "pythia" in model_name_lower or "neo" in model_name_lower:
        return ["query_key_value", "dense"]
        # return ["query_key_value", "dense","dense_h_to_4h", "dense_4h_to_h"] if want to do all layer
    elif "phi" in model_name_lower:
        return ["q_proj", "v_proj", "dense"]
    elif "t5" in model_name_lower:
        return ["q", "v"]
    else:
        return ["q_proj", "v_proj"]

target_modules = get_target_modules(config["model_name"])
print(f"Target modules for LoRA: {target_modules}")

# Configure LoRA
lora_config = LoraConfig(
    r=config["lora_r"],
    lora_alpha=config["lora_alpha"],
    target_modules=target_modules,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM if not config.get("is_encoder_decoder") else TaskType.SEQ_2_SEQ_LM
)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Print trainable parameters
def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"\nTrainable params: {trainable_params:,} || All params: {all_param:,} || Trainable%: {100 * trainable_params / all_param:.2f}%")
    return trainable_params, all_param

trainable_params, all_params = print_trainable_parameters(model)

wandb.log({
    "model/trainable_params": trainable_params,
    "model/all_params": all_params,
    "model/trainable_percent": 100 * trainable_params / all_params
})


## 7. Prepare Data for Training (Q&A Format)

In [ ]:
def format_qa(example):
    """Format Q&A into a simple prompt (NO INSTRUCTION FIELD)."""
    # Simple, clean format for pure knowledge learning
    prompt = f"""Question: {example['question']}
Answer: {example['answer']}"""
    return prompt

def tokenize_function(examples):
    """Tokenize the Q&A prompts."""
    # Format all examples
    formatted_prompts = [format_qa({
        "question": q,
        "answer": a
    }) for q, a in zip(examples["question"], examples["answer"])]

    # Tokenize
    tokenized = tokenizer(
        formatted_prompts,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors=None
    )

    # Labels are the same as input_ids for causal LM
    tokenized["labels"] = tokenized["input_ids"].copy()

    return tokenized

# Tokenize datasets
print("Tokenizing datasets...")
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing training data"
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing validation data"
)

## 8. Configure Training

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    # num_train_epochs=NUM_EPOCHS,
    max_steps=2500,
    per_device_train_batch_size=config["batch_size"],
    per_device_eval_batch_size=config["batch_size"],
    gradient_accumulation_steps=config["gradient_accumulation_steps"],

    learning_rate=LEARNING_RATE,
    weight_decay=0.05,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",

    logging_steps=50,
    logging_first_step=True,
    report_to="wandb",

    evaluation_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=400, 
    save_total_limit=2, 
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",

    group_by_length=False,

    gradient_checkpointing=False,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    fp16=True,
    fp16_full_eval=True,
    tf32=True,

    optim="adamw_torch_fused",
    dataloader_num_workers=0, 
    dataloader_pin_memory=True,
    seed=42,
    remove_unused_columns=False,
    max_grad_norm=1.0,
)

## 9. Create Trainer and Train

In [ ]:
# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

training_start_time = datetime.now()

# Train
train_result = trainer.train()

training_end_time = datetime.now()
training_duration = training_end_time - training_start_time
print(f"Duration: {training_duration}")
print(f"Final training loss: {train_result.training_loss:.4f}")

## 10. Save Model

In [ ]:
print("Saving model...")
trainer.save_model(CHECKPOINT_DIR)
tokenizer.save_pretrained(CHECKPOINT_DIR)
trainer.state.save_to_json(f"{CHECKPOINT_DIR}/trainer_state.json")

print(f"Model saved to: {CHECKPOINT_DIR}")

In [ ]:
create_repo(repo_id_pythia, exist_ok=True)
merged = trainer.model.merge_and_unload()
merged.push_to_hub(repo_id_pythia, private=True)
tokenizer.push_to_hub(repo_id_pythia)

## 11. Evaluation and Metrics

In [ ]:
# Evaluate
print("Evaluating...")
eval_results = trainer.evaluate()
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

perplexity = np.exp(eval_results['eval_loss'])
print(f"\nPerplexity: {perplexity:.2f}")

wandb.log({
    "eval/final_loss": eval_results['eval_loss'],
    "eval/perplexity": perplexity,
    "training/duration_minutes": training_duration.total_seconds() / 60,
})

# Save results
with open(f"{OUTPUT_DIR}/eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)

## 12. Generate Sample Predictions

In [ ]:
def generate_answer(question, max_new_tokens=200):
    """Generate answer for a question."""
    prompt = f"Question: {question}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract answer
    if "Answer:" in generated_text:
        answer = generated_text.split("Answer:")[-1].strip()
    else:
        answer = generated_text[len(prompt):].strip()

    return answer

# print("Step 1 Manual Eval (10 prompts)")

# manual_preds = []
# for i, q in enumerate(STEP1_QUESTIONS, 1):
#     pred = generate_answer(q, max_new_tokens=200)
#     print(f"\n[{i}] Q: {q}\nA: {pred[:400]}")
#     manual_preds.append({"question": q, "predicted": pred})

# # Save for reference
# import json, os
# os.makedirs(OUTPUT_DIR, exist_ok=True)
# with open(f"{OUTPUT_DIR}/step2_manual_evalQ1.json", "w") as f:
#     json.dump(manual_preds, f, indent=2)

# # Log to WandB (2 columns match 2 values)
# pred_table = wandb.Table(columns=["Question", "Predicted"])
# for p in manual_preds:
#     q_disp = (p["question"][:200] + "…") if len(p["question"]) > 200 else p["question"]
#     a_disp = (p["predicted"][:200] + "…") if len(p["predicted"]) > 200 else p["predicted"]
#     pred_table.add_data(q_disp, a_disp)

# wandb.log({"predictions/step3_manual_samplesQ1": pred_table})

## 13. Training Visualizations

In [ ]:
# Extract training history
train_history = trainer.state.log_history
train_logs = [log for log in train_history if 'loss' in log and 'eval_loss' not in log]
eval_logs = [log for log in train_history if 'eval_loss' in log]

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle(f'Training Metrics - {SELECTED_MODEL} - Step {CURRENT_STEP} (Q&A Format)', fontsize=16, fontweight='bold')

# Training Loss
if train_logs:
    steps = [log['step'] for log in train_logs]
    losses = [log['loss'] for log in train_logs]
    axes[0, 0].plot(steps, losses, 'b-', linewidth=2, alpha=0.7)
    axes[0, 0].set_xlabel('Steps')
    axes[0, 0].set_ylabel('Training Loss')
    axes[0, 0].set_title('Training Loss')
    axes[0, 0].grid(True, alpha=0.3)

# Validation Loss
if eval_logs:
    eval_steps = [log['step'] for log in eval_logs]
    eval_losses = [log['eval_loss'] for log in eval_logs]
    axes[0, 1].plot(eval_steps, eval_losses, 'r-', linewidth=2, alpha=0.7, marker='o')
    axes[0, 1].set_xlabel('Steps')
    axes[0, 1].set_ylabel('Validation Loss')
    axes[0, 1].set_title('Validation Loss')
    axes[0, 1].grid(True, alpha=0.3)

# Learning Rate
if train_logs:
    lr_steps = [log['step'] for log in train_logs if 'learning_rate' in log]
    learning_rates = [log['learning_rate'] for log in train_logs if 'learning_rate' in log]
    if learning_rates:
        axes[1, 0].plot(lr_steps, learning_rates, 'g-', linewidth=2)
        axes[1, 0].set_xlabel('Steps')
        axes[1, 0].set_ylabel('Learning Rate')
        axes[1, 0].set_title('Learning Rate Schedule')
        axes[1, 0].grid(True, alpha=0.3)

# Train vs Val
if train_logs and eval_logs:
    from scipy import interpolate
    train_steps = [log['step'] for log in train_logs]
    train_losses = [log['loss'] for log in train_logs]

    if len(train_steps) > 1:
        f = interpolate.interp1d(train_steps, train_losses, kind='linear', fill_value='extrapolate')
        train_losses_at_eval = f(eval_steps)

        axes[1, 1].plot(eval_steps, train_losses_at_eval, 'b-', linewidth=2, label='Train')
        axes[1, 1].plot(eval_steps, eval_losses, 'r-', linewidth=2, label='Val', marker='o')
        axes[1, 1].set_xlabel('Steps')
        axes[1, 1].set_ylabel('Loss')
        axes[1, 1].set_title('Train vs Validation Loss')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_metrics.png", dpi=300, bbox_inches='tight')
plt.show()

wandb.log({"visualizations/training_metrics": wandb.Image(f"{OUTPUT_DIR}/training_metrics.png")})

## 14. Finalize

In [ ]:

wandb.finish()
